# Real-World Scenario: Unsupervised Learning for Label Scarcity and Concept Drift

## Network Intrusion Detection with Evolving Threats

### Scenario Explanation

In modern cybersecurity environments, network intrusion detection systems face a critical challenge: attackers continuously evolve their tactics, creating novel attack patterns that have never been labeled or documented. Consider a large enterprise network where security analysts can only manually label a tiny fraction (less than 1%) of the millions of daily network traffic events, and new zero-day exploits emerge weekly that exhibit behaviors never seen before. Traditional supervised classifiers trained on historical labeled attacks fail catastrophically because they cannot recognize these novel threats—they are constrained to predict only the attack classes they were trained on, leaving the organization blind to emerging dangers. Unsupervised methods like Isolation Forest, DBSCAN, and KMeans excel in this environment because they learn the normal structure of network traffic without requiring labels, automatically flagging anomalous patterns that deviate from expected behavior regardless of whether those patterns have been previously categorized. The concrete benefit is transformative: these algorithms enable early detection of novel attack patterns (such as a new ransomware strain or advanced persistent threat) within hours of their first appearance, triggering immediate investigation before significant damage occurs, whereas supervised models would silently miss these threats until enough labeled examples accumulate for retraining—a delay that could span days or weeks. Furthermore, as network traffic patterns naturally drift over time due to infrastructure changes, software updates, or shifting user behavior, clustering algorithms adapt organically by continuously re-learning the evolving baseline, while supervised classifiers suffer from degrading accuracy and require expensive relabeling campaigns to maintain performance.

### Why Unsupervised Methods Outperform Supervised Classification

**Key Advantages:**

1. **Label Scarcity**: Clustering and anomaly detection algorithms (KMeans, DBSCAN, Isolation Forest) operate entirely without labels, making them ideal when labeling is expensive, time-consuming, or impossible for rare events.

2. **Concept Drift Resilience**: These methods continuously adapt to evolving data distributions by re-learning patterns from current data, whereas supervised models become stale and require periodic retraining with newly labeled data.

3. **Rare Event Detection**: Anomaly detection algorithms are specifically designed to identify outliers and rare patterns, while supervised classifiers struggle with class imbalance and often fail to learn minority classes effectively.

4. **Novel Pattern Discovery**: Unsupervised methods can detect completely new attack types or fraud schemes that have never been seen before, while supervised models are limited to predicting only the classes they were trained on.

5. **Operational Efficiency**: No need for continuous human annotation efforts, reducing operational costs and enabling real-time threat detection at scale.

### Algorithm Selection Guide

- **Isolation Forest**: Best for high-dimensional data with global anomalies; excels at detecting rare events in large datasets.
- **DBSCAN**: Ideal when anomalies form sparse regions; can discover arbitrary-shaped clusters and identify noise points.
- **KMeans**: Effective for discovering natural groupings in data; useful for identifying deviations from normal behavioral clusters.

### Concrete Benefit Example

**Early Detection of Novel Attack Patterns Without Labels:**

When a new ransomware variant begins encrypting files across the network, Isolation Forest immediately flags the unusual file access patterns (high frequency, specific file types, encryption-like behavior) as anomalies—even though this exact attack signature has never been labeled. Security teams receive alerts within minutes, enabling rapid containment before the attack spreads enterprise-wide. A supervised classifier would miss this entirely, as it has no training examples for this new ransomware family.

In [ ]:
# Demonstration: Simulating Novel Attack Detection
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN, KMeans
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Simulate normal network traffic (2 clusters representing different normal behaviors)
normal_traffic_1 = np.random.randn(300, 2) * 0.5 + np.array([2, 2])
normal_traffic_2 = np.random.randn(300, 2) * 0.5 + np.array([-2, -2])

# Simulate novel attack patterns (anomalies in different regions)
novel_attacks = np.random.randn(20, 2) * 0.3 + np.array([0, 4])

# Combine all data
X = np.vstack([normal_traffic_1, normal_traffic_2, novel_attacks])
true_labels = np.array([0]*300 + [0]*300 + [1]*20)  # 0=normal, 1=attack

print(f"Total network events: {len(X)}")
print(f"Normal traffic: {np.sum(true_labels == 0)} ({np.sum(true_labels == 0)/len(X)*100:.1f}%)")
print(f"Novel attacks: {np.sum(true_labels == 1)} ({np.sum(true_labels == 1)/len(X)*100:.1f}%)")

In [ ]:
# Apply Isolation Forest for anomaly detection
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_predictions = iso_forest.fit_predict(X)
iso_anomalies = iso_predictions == -1

print("\n=== Isolation Forest Results ===")
print(f"Detected anomalies: {np.sum(iso_anomalies)}")
print(f"True attacks detected: {np.sum((iso_anomalies) & (true_labels == 1))} / {np.sum(true_labels == 1)}")
print(f"Detection rate: {np.sum((iso_anomalies) & (true_labels == 1)) / np.sum(true_labels == 1) * 100:.1f}%")

In [ ]:
# Apply DBSCAN for density-based clustering
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X)
dbscan_anomalies = dbscan_labels == -1

print("\n=== DBSCAN Results ===")
print(f"Detected noise points (anomalies): {np.sum(dbscan_anomalies)}")
print(f"True attacks detected: {np.sum((dbscan_anomalies) & (true_labels == 1))} / {np.sum(true_labels == 1)}")
print(f"Detection rate: {np.sum((dbscan_anomalies) & (true_labels == 1)) / np.sum(true_labels == 1) * 100:.1f}%")
print(f"Number of clusters found: {len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)}")

In [ ]:
# Apply KMeans and use distance from cluster centers to detect anomalies
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X)
distances = np.min(kmeans.transform(X), axis=1)
threshold = np.percentile(distances, 95)  # Top 5% as anomalies
kmeans_anomalies = distances > threshold

print("\n=== KMeans-based Anomaly Detection ===")
print(f"Detected anomalies: {np.sum(kmeans_anomalies)}")
print(f"True attacks detected: {np.sum((kmeans_anomalies) & (true_labels == 1))} / {np.sum(true_labels == 1)}")
print(f"Detection rate: {np.sum((kmeans_anomalies) & (true_labels == 1)) / np.sum(true_labels == 1) * 100:.1f}%")

In [ ]:
# Visualization: Compare all three methods
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Ground Truth
axes[0, 0].scatter(X[true_labels == 0, 0], X[true_labels == 0, 1], 
                   c='blue', alpha=0.6, s=30, label='Normal Traffic')
axes[0, 0].scatter(X[true_labels == 1, 0], X[true_labels == 1, 1], 
                   c='red', alpha=0.8, s=80, marker='X', label='Novel Attacks', edgecolors='black')
axes[0, 0].set_title('Ground Truth: Novel Attack Patterns', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Feature 1 (e.g., Connection Rate)')
axes[0, 0].set_ylabel('Feature 2 (e.g., Data Transfer Volume)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Isolation Forest
axes[0, 1].scatter(X[~iso_anomalies, 0], X[~iso_anomalies, 1], 
                   c='blue', alpha=0.6, s=30, label='Normal')
axes[0, 1].scatter(X[iso_anomalies, 0], X[iso_anomalies, 1], 
                   c='red', alpha=0.8, s=80, marker='X', label='Detected Anomalies', edgecolors='black')
axes[0, 1].set_title('Isolation Forest Detection', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Feature 1')
axes[0, 1].set_ylabel('Feature 2')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# DBSCAN
unique_clusters = set(dbscan_labels)
colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_clusters) - (1 if -1 in unique_clusters else 0)))
color_idx = 0
for cluster in unique_clusters:
    if cluster == -1:
        axes[1, 0].scatter(X[dbscan_labels == cluster, 0], X[dbscan_labels == cluster, 1],
                          c='red', alpha=0.8, s=80, marker='X', label='Noise/Anomalies', edgecolors='black')
    else:
        axes[1, 0].scatter(X[dbscan_labels == cluster, 0], X[dbscan_labels == cluster, 1],
                          c=[colors[color_idx]], alpha=0.6, s=30, label=f'Cluster {cluster}')
        color_idx += 1
axes[1, 0].set_title('DBSCAN Clustering', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Feature 1')
axes[1, 0].set_ylabel('Feature 2')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# KMeans
axes[1, 1].scatter(X[~kmeans_anomalies, 0], X[~kmeans_anomalies, 1], 
                   c=kmeans.labels_[~kmeans_anomalies], cmap='viridis', alpha=0.6, s=30, label='Normal Clusters')
axes[1, 1].scatter(X[kmeans_anomalies, 0], X[kmeans_anomalies, 1], 
                   c='red', alpha=0.8, s=80, marker='X', label='Detected Anomalies', edgecolors='black')
axes[1, 1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                   c='yellow', s=200, marker='*', edgecolors='black', linewidths=2, label='Cluster Centers')
axes[1, 1].set_title('KMeans-based Detection', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Feature 1')
axes[1, 1].set_ylabel('Feature 2')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('unsupervised_anomaly_detection_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'unsupervised_anomaly_detection_comparison.png'")

### Key Takeaways

1. **No Labels Required**: All three algorithms successfully detected novel attack patterns without any prior labeling.

2. **Adaptability**: These methods can be retrained continuously on new data to adapt to concept drift in network traffic patterns.

3. **Early Warning System**: Anomalies are flagged immediately upon appearance, enabling proactive threat response.

4. **Complementary Strengths**: 
   - Isolation Forest excels at global outlier detection
   - DBSCAN identifies local density anomalies
   - KMeans provides interpretable cluster-based detection

5. **Real-World Impact**: In production systems, these algorithms enable security teams to detect zero-day exploits, novel malware, and advanced persistent threats hours or days before traditional signature-based or supervised methods.